# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.6 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile, glob, sys,math, random, collections, csv,base64,io
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID='task191'; CH=10; H=W=30; REAL_H=REAL_W=23
TASK_PATH=Path(os.environ.get('LOCAL_DATA','/mnt/data'))/f'{TASK_ID}.json'
if not TASK_PATH.exists():
    cs=list(Path('/kaggle/input').rglob(f'{TASK_ID}.json')) if Path('/kaggle/input').exists() else []
    TASK_PATH=cs[0] if cs else TASK_PATH
ONNX_PATH=Path.cwd()/f'{TASK_ID}.onnx'; SUBMISSION_PATH=Path.cwd()/'submission.zip'
FORBIDDEN_OPS={'Loop','Scan','NonZero','Unique','Script','Function'}
print('TASK_PATH=',TASK_PATH)

TASK_PATH= /kaggle/input/competitions/neurogolf-2026/task191.json


In [6]:

class Task191ConvStamp(nn.Module):
    def __init__(self):
        super().__init__()
        active=torch.zeros(1,1,H,W,dtype=torch.float32)
        active[:,:,0:REAL_H,0:REAL_W]=1.0
        self.register_buffer('active_region',active)
    def d4(self,m):
        tr=torch.transpose(m,2,3)
        return [m, torch.flip(m,[3]), torch.flip(m,[2]), torch.flip(m,[2,3]), tr, torch.flip(tr,[3]), torch.flip(tr,[2]), torch.flip(tr,[2,3])]
    def forward(self,x):
        x1=x[:,1:2,:,:]; x4=x[:,4:5,:,:]
        row_has=(x1.sum(dim=3,keepdim=True)>0.5).float(); col_has=(x1.sum(dim=2,keepdim=True)>0.5).float()
        bbox=row_has*col_has; src_one=x1; src_anchor=x4*bbox; add=torch.zeros_like(x1)
        for one_t,anchor_t in zip(self.d4(src_one),self.d4(src_anchor)):
            total=anchor_t.sum(dim=(2,3),keepdim=True)
            anchor_match=F.conv2d(x4,anchor_t,padding=29); one_conflict=F.conv2d(x4,one_t,padding=29)
            valid=((anchor_match>(total-0.5)) & (total>1.5) & (one_conflict<0.5)).float()
            fill=F.conv_transpose2d(valid,one_t,padding=29)
            add=torch.maximum(add,(fill>0.5).float())
        add=add*self.active_region*(1.0-x4)
        out=[]
        for c in range(CH):
            ch=x[:,c:c+1,:,:]
            if c==0: ch=ch*(1.0-add)
            elif c==1: ch=torch.maximum(ch,add)
            out.append(ch)
        return torch.cat(out,dim=1)
model=Task191ConvStamp().eval()


In [7]:
dummy=torch.zeros((1,CH,H,W),dtype=torch.float32); dummy[:,0,:,:]=1.0
torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],opset_version=17,do_constant_folding=True,dynamic_axes=None,dynamo=False)
m=onnx.load(str(ONNX_PATH)); onnx.checker.check_model(m); m=shape_inference.infer_shapes(m); onnx.save(m,str(ONNX_PATH))
print('saved',ONNX_PATH,ONNX_PATH.stat().st_size)

/tmp/ipykernel_16/1754810602.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,str(ONNX_PATH),input_names=['input'],output_names=['output'],opset_version=17,do_constant_folding=True,dynamic_axes=None,dynamo=False)


saved /kaggle/working/task191.onnx 44439


In [8]:
m=onnx.load(str(ONNX_PATH))
ops=dict(collections.Counter(n.op_type for n in m.graph.node))
forbidden=sorted(set(ops)&FORBIDDEN_OPS)
empty=[(n.name,n.op_type,list(n.input)) for n in m.graph.node if any(i=='' for i in n.input)]
print('ops',sorted(ops)); print('forbidden',forbidden); print('empty optional inputs',empty); print('size',ONNX_PATH.stat().st_size)
assert not forbidden and not empty and ONNX_PATH.stat().st_size<1440000

ops ['And', 'Cast', 'Concat', 'Constant', 'Conv', 'ConvTranspose', 'Greater', 'Less', 'Max', 'Mul', 'ReduceSum', 'Slice', 'Sub', 'Transpose']
forbidden []
empty optional inputs []
size 44439


In [9]:
if SUBMISSION_PATH.exists(): SUBMISSION_PATH.unlink()
with zipfile.ZipFile(SUBMISSION_PATH,'w',compression=zipfile.ZIP_DEFLATED) as z: z.write(ONNX_PATH,arcname=f'{TASK_ID}.onnx')
print('wrote',SUBMISSION_PATH,zipfile.ZipFile(SUBMISSION_PATH).namelist())

wrote /kaggle/working/submission.zip ['task191.onnx']
